# Analyzing Science Fiction via HathiTrust Extracted Features

This notebook analyzes the science fiction corpus built in notebook 01. It uses the page-level Extracted Features data to explore:

1. **Vocabulary analysis** -- most distinctive words in the sci-fi corpus
2. **Sentiment arcs** -- emotional trajectories across individual novels
3. **Vocabulary richness over time** -- how sci-fi language evolves across decades
4. **Cross-volume comparison** -- comparing word usage patterns across authors and eras

Requires: run notebook 01 first to download metadata and sample features.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## Load the Corpus

In [ ]:
# Load metadata
corpus_df = pd.read_csv('data/scifi_corpus_metadata.csv')
print(f"Corpus: {len(corpus_df)} volumes")

# Load downloaded EF sample files
SAMPLE_DIR = Path('data/ef_samples')
sample_files = sorted(SAMPLE_DIR.glob('*.json'))
print(f"Downloaded samples: {len(sample_files)} volumes")

# Parse all sample volumes
volumes = {}
for fp in sample_files:
    with open(fp) as f:
        data = json.load(f)
    htid = data.get('htid', fp.stem)
    volumes[htid] = data

print(f"Loaded {len(volumes)} volumes for analysis")

In [ ]:
def volume_to_tokenlist(vol_data):
    """Convert EF volume data to a flat token DataFrame."""
    rows = []
    pages = vol_data.get('features', {}).get('pages', [])
    
    for page in pages:
        if not page:
            continue
        seq = page.get('seq', 0)
        body = page.get('body')
        if not body:
            continue
        token_pos = body.get('tokenPosCount', {})
        if not token_pos:
            continue
        for token, pos_counts in token_pos.items():
            for pos, count in pos_counts.items():
                rows.append({
                    'page': int(seq) if seq else 0,
                    'token': token.lower(),
                    'pos': pos,
                    'count': count
                })
    
    return pd.DataFrame(rows)

def volume_title(vol_data):
    meta = vol_data.get('metadata', {})
    title = meta.get('title', 'Unknown')
    author = meta.get('contributor', {})
    if isinstance(author, dict):
        author = author.get('name', '')
    elif isinstance(author, list):
        author = author[0].get('name', '') if author else ''
    return f"{title.rstrip(' /')} ({author.split(',')[0]})"

## 1. Vocabulary Analysis

What are the most common words in the sci-fi corpus? How does the vocabulary compare to general English? We aggregate token counts across all downloaded volumes, excluding common stop words.

In [ ]:
# Aggregate word frequencies across all sample volumes
all_tokens = Counter()
per_volume_tokens = {}

for htid, vol_data in volumes.items():
    tl = volume_to_tokenlist(vol_data)
    if tl.empty:
        continue
    
    # Aggregate by token
    vol_counts = tl.groupby('token')['count'].sum()
    per_volume_tokens[htid] = vol_counts
    
    for token, count in vol_counts.items():
        all_tokens[token] += count

print(f"Total unique tokens: {len(all_tokens):,}")
print(f"Total token occurrences: {sum(all_tokens.values()):,}")

In [ ]:
# Filter to content words (exclude stop words and short tokens)
STOP_WORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
    'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall', 'can', 'not', 'no',
    'it', 'its', 'he', 'she', 'his', 'her', 'they', 'them', 'their',
    'we', 'us', 'our', 'you', 'your', 'i', 'me', 'my', 'this', 'that',
    'these', 'those', 'what', 'which', 'who', 'whom', 'how', 'when',
    'where', 'why', 'if', 'then', 'than', 'so', 'up', 'out', 'about',
    'into', 'over', 'after', 'before', 'between', 'under', 'through',
    'just', 'also', 'very', 'too', 'only', 'now', 'here', 'there',
    'all', 'each', 'every', 'both', 'few', 'more', 'most', 'other',
    'some', 'such', 'any', 'one', 'two', 'said', 'like', 'back',
    'down', 'still', 'even', 'well', 'way', 'own', 'same', 'man',
}

content_tokens = {
    tok: count for tok, count in all_tokens.items()
    if tok not in STOP_WORDS and len(tok) > 2 and tok.isalpha()
}

top_words = pd.Series(content_tokens).sort_values(ascending=False)
print("Top 40 content words in the sci-fi corpus:")
print(top_words.head(40))

In [ ]:
# Visualize top words
fig, ax = plt.subplots(figsize=(14, 5))
top_words.head(30).plot(kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Frequency')
ax.set_title('Top 30 Content Words in Science Fiction Corpus')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Sentiment Arcs

Using the AFINN sentiment lexicon, we plot emotional valence across each novel's pages. This follows Matthew Jockers' syuzhet methodology: assign sentiment scores to each page's tokens, then smooth with a rolling mean to reveal narrative arcs.

In [ ]:
from afinn import Afinn

afinn = Afinn()

# Build lookup dict
sentiment_dict = afinn._dict
print(f"AFINN lexicon: {len(sentiment_dict)} words")
print(f"Sample: {list(sentiment_dict.items())[:5]}")

In [ ]:
def page_sentiment(vol_data):
    """Compute sentiment score per page for a volume."""
    pages = vol_data.get('features', {}).get('pages', [])
    sentiments = []
    
    for page in pages:
        if not page:
            sentiments.append(0)
            continue
        body = page.get('body')
        if not body:
            sentiments.append(0)
            continue
        token_pos = body.get('tokenPosCount', {})
        score = 0
        for token, pos_counts in token_pos.items():
            token_lower = token.lower()
            if token_lower in sentiment_dict:
                token_count = sum(pos_counts.values())
                score += sentiment_dict[token_lower] * token_count
        sentiments.append(score)
    
    return pd.Series(sentiments)

In [ ]:
# Plot sentiment arcs for each volume in the sample
# Normalize page position to 0-1 so books of different lengths are comparable

n_vols = len(volumes)
cols = 3
rows = (n_vols + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

for idx, (htid, vol_data) in enumerate(volumes.items()):
    ax = axes[idx]
    sent = page_sentiment(vol_data)
    
    if len(sent) < 10:
        ax.set_title('(too short)', fontsize=9)
        continue
    
    # Normalize x-axis to 0-1
    x = np.linspace(0, 1, len(sent))
    
    # Rolling mean (window = 5% of book)
    window = max(3, len(sent) // 20)
    smoothed = sent.rolling(window, center=True).mean()
    
    ax.fill_between(x, 0, smoothed, alpha=0.3, color='steelblue')
    ax.plot(x, smoothed, color='steelblue', linewidth=1)
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_title(volume_title(vol_data)[:50], fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)

# Hide unused subplots
for idx in range(len(volumes), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Sentiment Arcs Across Science Fiction Novels', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Vocabulary Richness Over Time

Type-token ratio (TTR) measures vocabulary diversity: the number of unique words divided by total words. Higher TTR = more varied vocabulary. We compute this for each volume and plot it against publication date to see how sci-fi vocabulary evolves.

In [ ]:
richness = []

for htid, vol_data in volumes.items():
    tl = volume_to_tokenlist(vol_data)
    if tl.empty:
        continue
    
    total_tokens = tl['count'].sum()
    unique_tokens = tl['token'].nunique()
    meta = vol_data.get('metadata', {})
    pub_date = meta.get('pubDate', None)
    
    richness.append({
        'htid': htid,
        'title': volume_title(vol_data),
        'pub_date': pub_date,
        'total_tokens': total_tokens,
        'unique_tokens': unique_tokens,
        'ttr': unique_tokens / total_tokens if total_tokens > 0 else 0,
        'pages': len(vol_data.get('features', {}).get('pages', [])),
    })

richness_df = pd.DataFrame(richness)
richness_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TTR vs publication date
ax = axes[0]
ax.scatter(richness_df['pub_date'], richness_df['ttr'], s=60, color='steelblue', alpha=0.7)
for _, row in richness_df.iterrows():
    ax.annotate(row['title'][:20], (row['pub_date'], row['ttr']),
                fontsize=6, alpha=0.7, rotation=15)
ax.set_xlabel('Publication Year')
ax.set_ylabel('Type-Token Ratio')
ax.set_title('Vocabulary Richness Over Time')

# Volume length vs TTR
ax = axes[1]
ax.scatter(richness_df['total_tokens'], richness_df['ttr'], s=60, color='coral', alpha=0.7)
ax.set_xlabel('Total Tokens')
ax.set_ylabel('Type-Token Ratio')
ax.set_title('Vocabulary Richness vs. Book Length')

plt.tight_layout()
plt.show()

## 4. POS Distribution

The Extracted Features data includes part-of-speech tags for every token. We can compare the distribution of nouns, verbs, adjectives, etc. across volumes to see stylistic differences.

In [ ]:
# POS tag reference (Penn Treebank):
# NN/NNS = noun, NNP/NNPS = proper noun
# VB/VBD/VBG/VBN/VBP/VBZ = verb forms
# JJ/JJR/JJS = adjective
# RB/RBR/RBS = adverb

POS_GROUPS = {
    'Noun': ['NN', 'NNS'],
    'Proper Noun': ['NNP', 'NNPS'],
    'Verb': ['VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ'],
    'Adjective': ['JJ', 'JJR', 'JJS'],
    'Adverb': ['RB', 'RBR', 'RBS'],
}

pos_data = []
for htid, vol_data in volumes.items():
    tl = volume_to_tokenlist(vol_data)
    if tl.empty:
        continue
    
    total = tl['count'].sum()
    row = {'title': volume_title(vol_data)[:40]}
    
    for group_name, tags in POS_GROUPS.items():
        group_count = tl[tl['pos'].isin(tags)]['count'].sum()
        row[group_name] = group_count / total if total > 0 else 0
    
    pos_data.append(row)

pos_df = pd.DataFrame(pos_data).set_index('title')
pos_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
pos_df.plot(kind='barh', stacked=True, ax=ax,
            color=['steelblue', 'lightblue', 'coral', 'gold', 'mediumpurple'])
ax.set_xlabel('Proportion of Tokens')
ax.set_title('Part-of-Speech Distribution Across Science Fiction Volumes')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Cross-Volume Word Comparison

Which words are distinctive to specific volumes? We compute TF-IDF-like scores: words that appear frequently in one volume but rarely across the corpus.

In [ ]:
# Document frequency: how many volumes contain each word
doc_freq = Counter()
for htid, vol_counts in per_volume_tokens.items():
    for token in vol_counts.index:
        doc_freq[token] += 1

n_docs = len(per_volume_tokens)

def distinctive_words(htid, top_n=15):
    """Find words most distinctive to a volume using TF-IDF."""
    if htid not in per_volume_tokens:
        return pd.Series(dtype=float)
    
    vol_counts = per_volume_tokens[htid]
    total = vol_counts.sum()
    
    scores = {}
    for token, count in vol_counts.items():
        if not token.isalpha() or len(token) <= 2 or token in STOP_WORDS:
            continue
        tf = count / total
        idf = np.log(n_docs / (1 + doc_freq.get(token, 0)))
        scores[token] = tf * idf
    
    return pd.Series(scores).sort_values(ascending=False).head(top_n)

In [ ]:
# Show distinctive words for each volume
for htid, vol_data in volumes.items():
    title = volume_title(vol_data)
    words = distinctive_words(htid, top_n=10)
    if words.empty:
        continue
    print(f"\n{title}")
    print(f"  {', '.join(words.index)}")

## 6. Aggregate Sentiment by Decade

Does the overall emotional tone of science fiction shift across the 20th century? We compute mean sentiment per volume and group by decade.

In [ ]:
decade_sentiment = []

for htid, vol_data in volumes.items():
    meta = vol_data.get('metadata', {})
    pub_date = meta.get('pubDate')
    if not pub_date:
        continue
    
    sent = page_sentiment(vol_data)
    decade = (int(pub_date) // 10) * 10
    
    decade_sentiment.append({
        'htid': htid,
        'title': volume_title(vol_data),
        'pub_date': int(pub_date),
        'decade': decade,
        'mean_sentiment': sent.mean(),
        'std_sentiment': sent.std(),
    })

decade_df = pd.DataFrame(decade_sentiment)

fig, ax = plt.subplots(figsize=(12, 5))
decade_df.groupby('decade')['mean_sentiment'].mean().plot(
    kind='bar', ax=ax, color='steelblue', edgecolor='white'
)
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel('Decade')
ax.set_ylabel('Mean Sentiment Score')
ax.set_title('Mean Sentiment in Science Fiction by Decade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Next Steps

- **Scale up**: download features for more of the 5,811 volumes for more robust trends
- **Topic modeling**: use token frequencies to identify thematic clusters (space travel, dystopia, alien contact, etc.)
- **Comparison corpus**: download a matched non-SF fiction corpus from Underwood's [fiction metadata](http://data.analytics.hathitrust.org/genre/fiction_metadata.csv) to identify what's *distinctively* sci-fi vs. general fiction
- **Anthology segmentation**: Thompson presented at DH2020 on segmenting short stories from the 918 anthology volumes
- **ISFDB cross-reference**: use the `ISFDB ID` field in the expanded workset to pull genre sub-classifications (hard SF, cyberpunk, New Wave, etc.)
- **Network analysis**: build author similarity networks based on shared vocabulary